# Airbnb borough analysis

This notebook answers each question in its own cell. Run the setup cell first.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Load and prepare the Airbnb data
url = 'https://github.com/adelnehme/python-for-spreadsheet-users-webinar/blob/master/datasets/airbnb.csv?raw=true'
airbnb = pd.read_csv(url, index_col='Unnamed: 0')

airbnb[['borough', 'neighbourhood']] = airbnb['neighbourhood_full'].str.split(', ', n=1, expand=True)
airbnb['price'] = pd.to_numeric(airbnb['price'].str.replace('$', '', regex=False))
airbnb['listing_added'] = pd.to_datetime(airbnb['listing_added'])
room_type = airbnb['room_type'].str.strip().str.lower()
airbnb['room_type'] = room_type.replace({
    'entire home/apt': 'Entire place',
    'shared room': 'Shared Room',
    'home': 'Entire place'
})
# Combine every spelling/capitalization variant containing 'private' into one category.
airbnb.loc[room_type.str.contains('private', na=False), 'room_type'] = 'Private room'
airbnb = airbnb.drop_duplicates(subset='listing_id').copy()
airbnb.head()

## 1. Average price of listings by borough

In [ ]:
average_price_by_borough = (
    airbnb.groupby('borough', observed=True)
    .agg(average_price=('price', 'mean'))
    .reset_index()
)

display(average_price_by_borough)
sns.barplot(x='borough', y='average_price', data=average_price_by_borough, hue='borough', legend=False)
plt.title('Average Listing Price by Borough')
plt.xlabel('Borough')
plt.ylabel('Average price ($)')
plt.show()

## 2. Average availability in days by borough

In [ ]:
average_availability_by_borough = (
    airbnb.groupby('borough', observed=True)
    .agg(average_availability_days=('availability_365', 'mean'))
    .reset_index()
)

display(average_availability_by_borough)
sns.barplot(x='borough', y='average_availability_days', data=average_availability_by_borough, hue='borough', legend=False)
plt.title('Average Listing Availability by Borough')
plt.xlabel('Borough')
plt.ylabel('Average availability (days)')
plt.show()

## 3. Median price per room type in each borough

In [ ]:
median_price_by_room_and_borough = (
    airbnb.groupby(['borough', 'room_type'], observed=True)
    .agg(median_price=('price', 'median'))
    .reset_index()
)

display(median_price_by_room_and_borough)
sns.barplot(x='borough', y='median_price', hue='room_type', data=median_price_by_room_and_borough)
plt.title('Median Listing Price by Room Type and Borough')
plt.xlabel('Borough')
plt.ylabel('Median price ($)')
plt.legend(title='Room type')
plt.show()

## 4. Number of listings over time

In [ ]:
listings_over_time = (
    airbnb.assign(month=airbnb['listing_added'].dt.strftime('%Y-%m'))
    .groupby('month', observed=True)
    .agg(number_of_listings=('listing_id', 'count'))
    .reset_index()
)

sns.lineplot(x='month', y='number_of_listings', data=listings_over_time, marker='o')
plt.title('Number of Listings Added Over Time')
plt.xlabel('Month')
plt.ylabel('Number of listings')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()